# 00 · Load the match data

Reads every provided asset into numpy/pandas, aligned so that **row `i` = video frame `i` = row `i` of a submission**.
The loading code lives in `possession/load_data.py`; this notebook is where we look at it.

- Coordinates are in **metres**, origin at a pitch corner: x along the length (0–105.3), y across (0–68).
- `NaN` = not tracked (ball or player).
- First run parses the tracking (~3 s) and caches it in `cache/` (gitignored, never commit it).

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / 'possession').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
# data folder: $MTAH_DATA_DIR if set, else ./data inside the repo, else ../data next to it
# Update MTAH_DATA_DIR here when the gated data lives elsewhere.
DATA_CANDIDATES = []
if os.environ.get('MTAH_DATA_DIR'):
    DATA_CANDIDATES.append(Path(os.environ['MTAH_DATA_DIR']))
DATA_CANDIDATES.extend([ROOT / 'data', ROOT.parent / 'data'])
DATA = next((candidate.resolve() for candidate in DATA_CANDIDATES if (candidate / 'tactical').exists()), None)
if DATA is None:
    raise FileNotFoundError('Set MTAH_DATA_DIR in notebook 00 to the folder containing tactical/')
os.environ['MTAH_DATA_DIR'] = str(DATA)
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "starter"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from possession.load_data import load_match, summary, HOME, AWAY
from utils import draw_pitch

%load_ext autoreload
%autoreload 2
print("data dir:", os.environ["MTAH_DATA_DIR"])

data dir: <configured data directory>


## Load

In [2]:
m = load_match(data_dir=DATA)
summary(m)

frames            199,319  (graded 158,857)
pitch             105.3 x 68.0 m
track slots       46  home 24  away 22
ball seen         56.9% of all frames, 59.3% of graded
ball speed given  8.8% of graded frames
players tracked   mean 20.5 per graded frame
smart tagging     638 instances, 14 codes
homography valid  100.0%
public GT         {'A': 2577, 'D': 1469, 'H': 3454}


## What's in `m`

| attribute | shape | meaning |
|---|---|---|
| `ball_xy` | (N, 2) | ball position, metres |
| `ball_speed` | (N,) | as delivered, mostly NaN → recompute from positions |
| `players_xy` | (N, 46, 2) | every track slot (a tracklet, **not** a person) |
| `players_speed` | (N, 46) | m/s |
| `track_team` | (46,) | 0 = home Barcelona, 1 = away Athletic |
| `half`, `graded` | (N,) | 1/2 on scored frames, 0 otherwise |
| `tags` | DataFrame | Smart Tagging, `start_frame`/`end_frame` half-open, `team_side` = H/A/NaN (`side` is the Left/Central/Right tag) |
| `homography`, `homography_valid` | (N,3,3), (N,) | pitch (0–1) → pixels |
| `gt_public` | (N,) | H/A/D on frames 11185–18684, X elsewhere — **checking only, never tuning** |

In [3]:
{k: getattr(m, k).shape for k in ["ball_xy", "ball_speed", "players_xy", "players_speed",
                                   "track_team", "half", "graded", "homography", "gt_public"]}

{'ball_xy': (199319, 2),
 'ball_speed': (199319,),
 'players_xy': (199319, 46, 2),
 'players_speed': (199319, 46),
 'track_team': (46,),
 'half': (199319,),
 'graded': (199319,),
 'homography': (199319, 3, 3),
 'gt_public': (199319,)}

## A per-frame table (handy for quick looks)

## Frame-to-match-time utility

Frame time is part of the shared data interface, not a possession heuristic. The helper adds:

- video_time_s: exact tactical video time at the supplied frame rate;
- half_elapsed_s: signed time from the verified half kickoff;
- match_clock_s: kickoff-anchored analytical clock, with 0:00 at first-half kickoff and 45:00 at second-half kickoff;
- match_clock_status: makes the graded second-half pre-kickoff interval explicit.

The analytical clock is for reproducible lookup and plot annotation. It is not a claim about the referee’s displayed stoppage-time clock.

In [4]:
from possession.baseline_pipeline import add_match_time_columns

time_index = add_match_time_columns(pd.DataFrame({'frame': np.arange(len(m.half))}))
display(time_index.loc[time_index.match_half.ne(time_index.match_half.shift()), ['frame', 'match_half', 'video_time_s', 'half_elapsed_s', 'match_clock_s', 'match_clock_status']])

,frame,match_half,video_time_s,half_elapsed_s,match_clock_s,match_clock_status
0,0,1H,0.00,-447.40,-447.40,in-play timeline
110808,110808,2H,4432.32,-117.36,2582.64,2H pre-kickoff


In [5]:
frames = pd.DataFrame({
    "half": m.half,
    "ball_x": m.ball_xy[:, 0], "ball_y": m.ball_xy[:, 1],
    "ball_seen": ~np.isnan(m.ball_xy[:, 0]),
    "n_home": (~np.isnan(m.players_xy[:, m.track_team == HOME, 0])).sum(1),
    "n_away": (~np.isnan(m.players_xy[:, m.track_team == AWAY, 0])).sum(1),
    "gt": m.gt_public,
})
frames.loc[11185:11195]

,half,ball_x,ball_y,ball_seen,n_home,n_away,gt
11185,1,NaN,NaN,False,10,10,A
11186,1,NaN,NaN,False,10,10,A
11187,1,53.103848,32.701199,True,10,10,A
11188,1,53.763023,32.877319,True,10,10,A
11189,1,54.430626,33.056839,True,10,10,A
11190,1,55.104546,33.238400,True,10,10,A
11191,1,55.781620,33.421322,True,10,10,A
11192,1,56.460804,33.604919,True,10,10,A
11193,1,57.138943,33.788521,True,11,10,A
11194,1,57.813915,33.972122,True,11,10,A


## Downstream audit views

This notebook intentionally stops after loading the shared match object, its frame table, and the frame-to-time utility.
Pitch views, missing-ball analysis, public-window checks, and Smart Tagging conclusions live in **01 — Problem, data, and findings**.
To reuse the submission, update only `MTAH_DATA_DIR` (or the fallback data path) here; the later notebooks consume the same interface.


## Organiser handoff

**Problem → idea → construction → evaluation.** Possession truth is not directly observed, so every later stage must consume the same frame-aligned tracking/event interface. This notebook constructs that reproducible input boundary; notebooks 01–06 derive observations, scenarios, anchors, and transition evidence, while 08 runs the final full-match pipeline and 09 compares alternatives. Ground truth is read only by the evaluation notebooks after prediction.